# 06 - Palmgren-Miner Cumulative Damage

Demonstrates the Palmgren-Miner linear damage accumulation rule for variable amplitude loading.

$$D = \sum_{i=1}^{k} \frac{n_i}{N_i}$$

Failure criterion: D ≥ 1.0 (or custom limit)

In [ ]:
from weldfatigue.fatigue.damage import PalmgrenMiner
from weldfatigue.fatigue.sn_curve import SNCurve
from weldfatigue.reporting.plots import FatiguePlots
import matplotlib.pyplot as plt

## Define Load Spectrum

In [ ]:
# Typical automotive load spectrum: (stress_range_MPa, n_cycles)
spectrum = [
    (140.0, 50_000),
    (120.0, 200_000),
    (100.0, 500_000),
    (80.0,  1_000_000),
    (60.0,  3_000_000),
    (40.0,  8_000_000),
]

print(f"{'Block':>5}  {'Δσ [MPa]':>10}  {'n_i':>12}")
print("-" * 30)
for i, (ds, n) in enumerate(spectrum):
    print(f"{i+1:>5}  {ds:>10.0f}  {n:>12,d}")
print(f"\nTotal cycles: {sum(n for _, n in spectrum):,d}")

## Compute Miner Damage

In [ ]:
sn = SNCurve(fat_class=80, material_type="steel", variable_amplitude=True)
miner = PalmgrenMiner(sn, damage_limit=1.0)

result = miner.compute_damage(spectrum)

print(f"{'Block':>5}  {'Δσ':>6}  {'n_i':>10}  {'N_i':>12}  {'D_i':>10}")
print("-" * 50)
for i, (ds, n) in enumerate(spectrum):
    n_allow = sn.cycles_to_failure(ds)
    d_i = result.damage_per_block[i]
    print(f"{i+1:>5}  {ds:>6.0f}  {n:>10,d}  {n_allow:>12.0f}  {d_i:>10.6f}")

print(f"\nTotal damage D = {result.total_damage:.4f}")
print(f"Status: {result.status}")

## Damage Histogram

In [ ]:
stress_ranges = [ds for ds, _ in spectrum]
fig = FatiguePlots.damage_histogram_plotly(result.damage_per_block, stress_ranges)
fig.show()

## Equivalent Constant Amplitude Stress

In [ ]:
eq_stress = miner.equivalent_stress_range(spectrum)
total_n = sum(n for _, n in spectrum)
print(f"Equivalent stress range: {eq_stress:.2f} MPa")
print(f"Total cycles: {total_n:,d}")
print(f"This means: {total_n:,d} cycles at Δσ = {eq_stress:.1f} MPa gives the same damage")

## Effect of Damage Limit

In [ ]:
for limit in [0.5, 0.8, 1.0]:
    m = PalmgrenMiner(sn, damage_limit=limit)
    r = m.compute_damage(spectrum)
    print(f"D_limit = {limit:.1f} → D = {r.total_damage:.4f} → {r.status}")